<a href="https://colab.research.google.com/github/vivek28n/Medical-RAG-Hallucination-Detection/blob/main/notebooks/otebook_06_Confidence_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# NOTEBOOK 06 - CONFIDENCE SCORING
# ============================================

print("Notebook 6 - Confidence Scoring")
print("Starting setup...")

Notebook 6 - Confidence Scoring
Starting setup...


In [ ]:
# ============================================
# STEP 2 - CHECK NOTEBOOK 5 OUTPUTS
# ============================================

print("Checking available variables from Notebook 5...")

required_variables = [
    "semantic_similarity",
    "nli_entailment",
    "retrieval_quality",
    "consistency"
]

for var in required_variables:
    if var in globals():
        print(f"✓ {var} is available")
    else:
        print(f"✗ {var} is NOT available")

Checking available variables from Notebook 5...
✗ semantic_similarity is NOT available
✗ nli_entailment is NOT available
✗ retrieval_quality is NOT available
✗ consistency is NOT available


In [ ]:
# ============================================
# STEP 3 - INSPECT AVAILABLE NOTEBOOK 5 OUTPUTS
# ============================================

print("Available variables related to Notebook 5:")

variable_names = list(globals().keys())

for name in variable_names:
    if any(keyword in name.lower() for keyword in
           ["similar", "nli", "entail", "retriev", "consist", "halluc", "support"]):
        print("✓", name)

Available variables related to Notebook 5:


In [3]:
# ============================================
# STEP 3 - CLAIM-LEVEL CONSISTENCY
# ============================================

def split_into_claims(answer):
    """
    Split the generated answer into individual claims/sentences.
    """

    claims = [
        sentence.strip()
        for sentence in answer.replace("\n", " ").split(".")
        if sentence.strip()
    ]

    return claims


def calculate_consistency(answer, retrieved_documents, entailment_threshold=0.50):
    """
    Measure how many answer claims are supported by
    at least one retrieved document.

    For each claim:
    - Compare it with every retrieved document using NLI.
    - Take the maximum entailment score.
    - A claim is considered supported if its entailment
      score reaches the initial threshold.

    Consistency =
    supported claims / total claims
    """

    claims = split_into_claims(answer)

    if not claims or not retrieved_documents:
        return 0.0

    supported_claims = 0

    for claim in claims:

        max_entailment = 0.0

        for doc in retrieved_documents:

            entailment, neutral, contradiction = nli_score(
                doc["text"],
                claim
            )

            max_entailment = max(
                max_entailment,
                entailment
            )

        if max_entailment >= entailment_threshold:
            supported_claims += 1

    consistency = supported_claims / len(claims)

    return float(consistency)


print("✅ Consistency function created")

✅ Consistency function created


In [4]:
# ============================================
# STEP 4 - CONFIDENCE LEVEL
# ============================================

def get_confidence_level(confidence_score):
    """
    Convert the numerical confidence score
    into a qualitative confidence level.
    """

    if confidence_score >= 0.80:
        return "HIGH"

    elif confidence_score >= 0.60:
        return "MEDIUM"

    else:
        return "LOW"


print("✅ Confidence level function created")

✅ Confidence level function created


In [5]:
# ============================================
# STEP 5 - COMBINE ALL CONFIDENCE COMPONENTS
# ============================================

def calculate_final_confidence(
    semantic_similarity,
    nli_entailment,
    retrieval_quality,
    consistency
):
    """
    Calculate the final confidence score using
    all four evidence-based components.
    """

    confidence_score = calculate_confidence(
        semantic_similarity=semantic_similarity,
        nli_entailment=nli_entailment,
        retrieval_quality=retrieval_quality,
        consistency=consistency
    )

    confidence_level = get_confidence_level(confidence_score)

    return {
        "confidence_score": confidence_score,
        "confidence_level": confidence_level
    }


print("✅ Final confidence calculator created")

✅ Final confidence calculator created


In [7]:
# ============================================
# STEP 6A - NOTEBOOK 5 → NOTEBOOK 6
# ============================================

def build_confidence_from_notebook5(
    notebook5_result,
    retrieval_quality,
    consistency
):
    """
    Connect Notebook 5 hallucination-detection output
    with Notebook 6 confidence scoring.

    Required Notebook 5 outputs:
    - semantic_similarity
    - nli_entailment
    - support_score
    - decision
    """

    semantic_similarity = notebook5_result["semantic_similarity"]
    nli_entailment = notebook5_result["nli_entailment"]

    confidence_result = calculate_final_confidence(
        semantic_similarity=semantic_similarity,
        nli_entailment=nli_entailment,
        retrieval_quality=retrieval_quality,
        consistency=consistency
    )

    return {
        "semantic_similarity": semantic_similarity,
        "nli_entailment": nli_entailment,
        "support_score": notebook5_result["support_score"],
        "retrieval_quality": retrieval_quality,
        "consistency": consistency,
        "confidence_score": confidence_result["confidence_score"],
        "confidence_level": confidence_result["confidence_level"],
        "decision": notebook5_result["decision"]
    }


print("✅ Notebook 5 → Notebook 6 integration function created")

✅ Notebook 5 → Notebook 6 integration function created


In [8]:
# ============================================
# STEP 6B - NOTEBOOK 5 DEPENDENCIES
# ============================================

!pip install -q pymupdf langchain-text-splitters sentence-transformers transformers torch faiss-cpu google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.5 MB/s eta 0:00:00


In [9]:
# ============================================
# STEP 6B - IMPORTS
# ============================================

import os
import re
import numpy as np
import torch
import fitz
import faiss

from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from langchain_text_splitters import RecursiveCharacterTextSplitter

from google import genai
from google.colab import userdata

print("✅ All required libraries imported")

✅ All required libraries imported


In [11]:
# ============================================
# STEP 6C-1 - CHECK REPOSITORY
# ============================================

import os

print("Current directory:")
print(os.getcwd())

print("\n/content contents:")
print(os.listdir("/content"))

Current directory:
/content

/content contents:
['.config', 'sample_data']


In [12]:
# ============================================
# STEP 6C-2 - CLONE PROJECT REPOSITORY
# ============================================

!git clone https://github.com/vivek28n/Medical-RAG-Hallucination-Detection.git

print("✅ Repository cloned successfully")

Cloning into 'Medical-RAG-Hallucination-Detection'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 98 (delta 52), reused 34 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 1.42 MiB | 8.14 MiB/s, done.
Resolving deltas: 100% (52/52), done.
✅ Repository cloned successfully


In [13]:
# ============================================
# STEP 6C-3 - VERIFY SOURCE PDF
# ============================================

PDF_PATH = (
    "/content/Medical-RAG-Hallucination-Detection/"
    "dataset/raw/niddk_guiding_principles_diabetes.pdf"
)

print("PDF exists:", os.path.exists(PDF_PATH))
print("PDF path:", PDF_PATH)

PDF exists: True
PDF path: /content/Medical-RAG-Hallucination-Detection/dataset/raw/niddk_guiding_principles_diabetes.pdf


In [14]:
# ============================================
# STEP 6C-4 - PDF EXTRACTION & CHUNKING
# ============================================

pdf = fitz.open(PDF_PATH)

pages = []

for page_number, page in enumerate(pdf, start=1):
    text = page.get_text("text").strip()

    if text:
        pages.append({
            "page": page_number,
            "text": text
        })

pdf.close()

print(f"📄 Pages with text: {len(pages)}")


# Same configuration as Notebook 3
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for page in pages:
    page_chunks = text_splitter.split_text(page["text"])

    for chunk in page_chunks:
        chunks.append({
            "text": chunk,
            "page": page["page"]
        })

print(f"🧩 Total chunks: {len(chunks)}")

📄 Pages with text: 83
🧩 Total chunks: 279


In [15]:
# ============================================
# STEP 6D - EMBEDDINGS & FAISS INDEX
# ============================================

# Load the same embedding model used in Notebook 3/5
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract chunk text
chunk_texts = [chunk["text"] for chunk in chunks]

# Generate normalized embeddings
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Convert to NumPy float32 for FAISS
chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype="float32"
)

print(f"🧠 Embedding shape: {chunk_embeddings.shape}")


# Create FAISS index
embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatL2(embedding_dimension)

faiss_index.add(chunk_embeddings)

print(f"🔎 FAISS index size: {faiss_index.ntotal}")
print(f"📐 Embedding dimension: {embedding_dimension}")
print("✅ Embeddings and FAISS index ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

🧠 Embedding shape: (279, 384)
🔎 FAISS index size: 279
📐 Embedding dimension: 384
✅ Embeddings and FAISS index ready


In [ ]:
# ============================================
# STEP 3 - DEFINE NOTEBOOK 5 OUTPUTS
# ============================================

print("Notebook 5 output structure identified.")

NOTEBOOK_5_OUTPUTS = [
    "semantic_similarity",
    "nli_entailment",
    "support_score",
    "decision"
]

for output in NOTEBOOK_5_OUTPUTS:
    print("✓", output)

Notebook 5 output structure identified.
✓ semantic_similarity
✓ nli_entailment
✓ support_score
✓ decision


In [16]:
# ============================================
# STEP 6E - DOCUMENT RETRIEVAL
# ============================================

def retrieve_documents(query, top_k=4):
    """
    Retrieve the most relevant documents from FAISS.

    Returns:
    - retrieved documents
    - cosine similarity scores
    """

    # Create normalized query embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # Search FAISS
    distances, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []
    similarities = []

    for distance, index in zip(distances[0], indices[0]):

        if index == -1:
            continue

        # Because embeddings are normalized:
        # L2 distance = 2 - 2*cosine_similarity
        cosine_similarity = 1.0 - (float(distance) / 2.0)

        retrieved_documents.append({
            "text": chunks[index]["text"],
            "page": chunks[index]["page"],
            "score": cosine_similarity
        })

        similarities.append(cosine_similarity)

    return retrieved_documents, similarities


print("✅ Retrieval function created")

✅ Retrieval function created


In [17]:
# ============================================
# STEP 6F - NLI MODEL SETUP
# ============================================

NLI_MODEL = "cross-encoder/nli-deberta-v3-base"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)

nli_model = AutoModelForSequenceClassification.from_pretrained(
    NLI_MODEL
)

nli_model.eval()

print(f"🧠 NLI model loaded: {NLI_MODEL}")
print("✅ NLI model ready")

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

🧠 NLI model loaded: cross-encoder/nli-deberta-v3-base
✅ NLI model ready


In [18]:
# ============================================
# STEP 6F - NLI SCORING FUNCTION
# ============================================

def nli_score(premise, hypothesis):
    """
    Calculate NLI probabilities between evidence and claim/answer.

    Returns:
    - entailment
    - neutral
    - contradiction
    """

    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # Model label order used in Notebook 5
    contradiction_score = float(probabilities[0])
    entailment_score = float(probabilities[1])
    neutral_score = float(probabilities[2])

    return (
        entailment_score,
        neutral_score,
        contradiction_score
    )


print("✅ NLI scoring function created")

✅ NLI scoring function created


In [19]:
# ============================================
# STEP 6G-1 - GEMINI CLIENT
# ============================================

API_KEY = userdata.get("Vivek28n")

client = genai.Client(api_key=API_KEY)

print("✅ Gemini client ready")

✅ Gemini client ready


In [20]:
# ============================================
# STEP 6G-2 - GROUNDED RAG ANSWER GENERATION
# ============================================

def generate_rag_answer(question, retrieved_documents):
    """
    Generate an answer using only the retrieved evidence.
    """

    if not retrieved_documents:
        return "I don't have enough evidence in the provided document."

    context_parts = []

    for rank, doc in enumerate(retrieved_documents, start=1):
        context_parts.append(
            f"[Source {rank} | Page {doc['page']}]\n"
            f"{doc['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a medical document question-answering assistant.

Answer the user's question using ONLY the provided
document evidence.

Do not add information that is not supported by the evidence.
If the evidence is insufficient, clearly say that there
is not enough evidence to answer the question.

Include source page numbers in the answer when possible.

USER QUESTION:
{question}

DOCUMENT EVIDENCE:
{context}
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    return response.text.strip()


print("✅ Grounded RAG answer function created")

✅ Grounded RAG answer function created


In [21]:
# ============================================
# STEP 6H-1 - SEMANTIC SIMILARITY
# ============================================

def semantic_similarity(answer, retrieved_documents):
    """
    Calculate the maximum semantic similarity between
    the generated answer and retrieved evidence.
    """

    if not answer or not retrieved_documents:
        return 0.0, []

    answer_embedding = embedding_model.encode(
        answer,
        normalize_embeddings=True
    )

    document_texts = [
        doc["text"] for doc in retrieved_documents
    ]

    document_embeddings = embedding_model.encode(
        document_texts,
        normalize_embeddings=True
    )

    similarities = np.dot(
        document_embeddings,
        answer_embedding
    )

    max_similarity = float(
        np.max(similarities)
    )

    return max_similarity, similarities


print("✅ Semantic similarity function created")

✅ Semantic similarity function created


In [22]:
# ============================================
# STEP 6H-2 - NLI AGAINST RETRIEVED DOCUMENTS
# ============================================

def check_nli_against_documents(
    answer,
    retrieved_documents
):
    """
    Check whether each retrieved document entails,
    contradicts, or is neutral toward the answer.
    """

    results = []

    for rank, doc in enumerate(
        retrieved_documents,
        start=1
    ):

        entailment, neutral, contradiction = nli_score(
            doc["text"],
            answer
        )

        results.append({
            "rank": rank,
            "page": doc.get("page"),
            "entailment": entailment,
            "neutral": neutral,
            "contradiction": contradiction
        })

    return results


print("✅ Document-level NLI function created")

✅ Document-level NLI function created


In [23]:
# ============================================
# STEP 6H-3 - SUPPORT SCORE & HALLUCINATION DECISION
# ============================================

SUPPORT_THRESHOLD = 0.60
CONTRADICTION_THRESHOLD = 0.50


def get_max_entailment(nli_results):
    if not nli_results:
        return 0.0

    return max(
        result["entailment"]
        for result in nli_results
    )


def calculate_support_score(
    similarity_score,
    entailment_score
):
    return (
        0.50 * similarity_score +
        0.50 * entailment_score
    )


def detect_hallucination(
    support_score,
    nli_results
):
    if not nli_results:
        return "POTENTIAL HALLUCINATION"

    max_contradiction = max(
        result["contradiction"]
        for result in nli_results
    )

    if max_contradiction >= CONTRADICTION_THRESHOLD:
        return "CONTRADICTED"

    elif support_score >= SUPPORT_THRESHOLD:
        return "SUPPORTED"

    else:
        return "POTENTIAL HALLUCINATION"


print("✅ Hallucination decision functions created")

✅ Hallucination decision functions created


In [26]:
# ============================================
# STEP 6I - END-TO-END RAG TEST
# ============================================

question = "What are the risk factors for diabetes?"

# 1. Retrieve relevant documents
retrieved_documents, similarities = retrieve_documents(
    question,
    top_k=4
)

print("🔎 Retrieved Documents:")
for i, doc in enumerate(retrieved_documents, start=1):
    print(
        f"Rank {i} | Page {doc['page']} | "
        f"Similarity: {doc['score']:.4f}"
    )

print("\n" + "=" * 60)

# 2. Generate grounded answer
answer = generate_rag_answer(
    question,
    retrieved_documents
)

print("🤖 Generated Answer:")
print(answer)

print("\n" + "=" * 60)

# 3. Run Notebook 5 hallucination analysis
notebook5_result = analyze_answer(
    answer,
    retrieved_documents
)

print("🧪 Notebook 5 Analysis:")
print(
    f"Semantic Similarity : "
    f"{notebook5_result['semantic_similarity']:.4f}"
)

print(
    f"NLI Entailment      : "
    f"{notebook5_result['nli_entailment']:.4f}"
)

print(
    f"Support Score       : "
    f"{notebook5_result['support_score']:.4f}"
)

print(
    f"Decision             : "
    f"{notebook5_result['decision']}"
)

🔎 Retrieved Documents:
Rank 1 | Page 5 | Similarity: 0.6716
Rank 2 | Page 9 | Similarity: 0.6615
Rank 3 | Page 5 | Similarity: 0.6370
Rank 4 | Page 1 | Similarity: 0.6251

🤖 Generated Answer:
Based on the provided documents, the risk factors for type 2 diabetes include:

* **Prediabetes:** Having blood glucose levels that are higher than normal, but not high enough to be characterized as diabetes [Source 1 | Page 5].
* **Age:** Risk increases with age [Source 2 | Page 9].
* **Overweight or obesity:** A body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source 2 | Page 9].
* **Family history of diabetes:** Having a parent or sibling with diabetes [Source 2 | Page 9].
* **Race/ethnicity:** Being a member of a high-risk population, including African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, and Pacific Islander American [Source 2 | Page 9].
* **History of GDM** (gestational diabetes mellitus) [Source 2 | Page 9].
* **Physical inactivity*

In [27]:
# ============================================
# STEP 6J-1 - ACTUAL RETRIEVAL QUALITY
# ============================================

retrieval_quality = calculate_retrieval_quality(
    similarities
)

print(
    f"🔎 Retrieval Quality: "
    f"{retrieval_quality:.4f}"
)

🔎 Retrieval Quality: 0.6488


In [28]:
# ============================================
# STEP 6J-2 - ACTUAL CONSISTENCY
# ============================================

consistency = calculate_consistency(
    answer,
    retrieved_documents,
    entailment_threshold=0.50
)

print(
    f"🧩 Consistency: "
    f"{consistency:.4f}"
)

🧩 Consistency: 0.0000


In [29]:
# ============================================
# STEP 6J-3 - DEBUG CONSISTENCY
# ============================================

claims = split_into_claims(answer)

print(f"📝 Total claims detected: {len(claims)}")
print("=" * 70)

for i, claim in enumerate(claims, start=1):

    max_entailment = 0.0
    best_page = None

    for doc in retrieved_documents:

        entailment, neutral, contradiction = nli_score(
            doc["text"],
            claim
        )

        if entailment > max_entailment:
            max_entailment = entailment
            best_page = doc["page"]

    print(f"\nClaim {i}:")
    print(claim[:250])

    print(
        f"Max Entailment: {max_entailment:.4f} "
        f"| Best Page: {best_page}"
    )

📝 Total claims detected: 9

Claim 1:
Based on the provided documents, the risk factors for type 2 diabetes include:  * **Prediabetes:** Having blood glucose levels that are higher than normal, but not high enough to be characterized as diabetes [Source 1 | Page 5]
Max Entailment: 0.0273 | Best Page: 9

Claim 2:
* **Age:** Risk increases with age [Source 2 | Page 9]
Max Entailment: 0.0321 | Best Page: 9

Claim 3:
* **Overweight or obesity:** A body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source 2 | Page 9]
Max Entailment: 0.0114 | Best Page: 9

Claim 4:
* **Family history of diabetes:** Having a parent or sibling with diabetes [Source 2 | Page 9]
Max Entailment: 0.0465 | Best Page: 9

Claim 5:
* **Race/ethnicity:** Being a member of a high-risk population, including African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, and Pacific Islander American [Source 2 | Page 9]
Max Entailment: 0.0069 | Best Page: 9

Claim 6:
* **History of GDM

In [30]:
# ============================================
# STEP 6J-4 - VERIFY NLI LABEL MAPPING
# ============================================

print("NLI model label mapping:")
print(nli_model.config.id2label)

print("\nNLI model number of labels:")
print(nli_model.config.num_labels)

NLI model label mapping:
{0: 'contradiction', 1: 'entailment', 2: 'neutral'}

NLI model number of labels:
3


In [31]:
# ============================================
# STEP 6J-5 - CONTROLLED NLI TEST
# ============================================

premise = (
    "People who are overweight or obese have an increased "
    "risk of developing type 2 diabetes."
)

hypothesis = (
    "Overweight or obesity is a risk factor for type 2 diabetes."
)

entailment, neutral, contradiction = nli_score(
    premise,
    hypothesis
)

print(f"Entailment   : {entailment:.4f}")
print(f"Neutral      : {neutral:.4f}")
print(f"Contradiction: {contradiction:.4f}")

Entailment   : 0.9979
Neutral      : 0.0020
Contradiction: 0.0001


In [32]:
# ============================================
# STEP 6J-6 - IMPROVED CLAIM CONSISTENCY
# ============================================

def calculate_consistency(
    answer,
    retrieved_documents,
    entailment_threshold=0.50,
    similarity_threshold=0.50
):
    """
    Calculate claim-level consistency.

    For each claim:
    1. Find the most semantically relevant retrieved evidence.
    2. Run NLI between that evidence and the claim.
    3. Mark the claim as supported if the NLI entailment
       score reaches the threshold.

    Consistency =
    supported claims / total claims
    """

    claims = split_into_claims(answer)

    if not claims or not retrieved_documents:
        return 0.0

    supported_claims = 0

    for claim in claims:

        # Embed claim
        claim_embedding = embedding_model.encode(
            claim,
            normalize_embeddings=True
        )

        best_similarity = -1.0
        best_document = None

        # Find the most semantically relevant evidence
        for doc in retrieved_documents:

            doc_embedding = embedding_model.encode(
                doc["text"],
                normalize_embeddings=True
            )

            similarity = float(
                np.dot(doc_embedding, claim_embedding)
            )

            if similarity > best_similarity:
                best_similarity = similarity
                best_document = doc

        # Only perform NLI when evidence is reasonably relevant
        if (
            best_document is not None
            and best_similarity >= similarity_threshold
        ):

            entailment, neutral, contradiction = nli_score(
                best_document["text"],
                claim
            )

            if entailment >= entailment_threshold:
                supported_claims += 1

    consistency = supported_claims / len(claims)

    return float(consistency)


print("✅ Improved claim-level consistency function created")

✅ Improved claim-level consistency function created


In [33]:
# ============================================
# STEP 6J-7 - RECALCULATE CONSISTENCY
# ============================================

consistency = calculate_consistency(
    answer,
    retrieved_documents,
    entailment_threshold=0.50,
    similarity_threshold=0.50
)

print(f"🧩 Improved Consistency: {consistency:.4f}")

🧩 Improved Consistency: 0.0000


In [34]:
# ============================================
# STEP 6J-8 - DETAILED CONSISTENCY DEBUG
# ============================================

claims = split_into_claims(answer)

print(f"📝 Total claims: {len(claims)}")
print("=" * 80)

for i, claim in enumerate(claims, start=1):

    claim_embedding = embedding_model.encode(
        claim,
        normalize_embeddings=True
    )

    best_similarity = -1.0
    best_document = None

    for doc in retrieved_documents:

        doc_embedding = embedding_model.encode(
            doc["text"],
            normalize_embeddings=True
        )

        similarity = float(
            np.dot(doc_embedding, claim_embedding)
        )

        if similarity > best_similarity:
            best_similarity = similarity
            best_document = doc

    if best_document is not None:

        entailment, neutral, contradiction = nli_score(
            best_document["text"],
            claim
        )

        print(f"\nClaim {i}:")
        print(claim[:200])

        print(
            f"Best semantic similarity : "
            f"{best_similarity:.4f}"
        )

        print(
            f"Best evidence page       : "
            f"{best_document['page']}"
        )

        print(
            f"NLI entailment           : "
            f"{entailment:.4f}"
        )

        print(
            f"NLI neutral              : "
            f"{neutral:.4f}"
        )

        print(
            f"NLI contradiction        : "
            f"{contradiction:.4f}"
        )

📝 Total claims: 9

Claim 1:
Based on the provided documents, the risk factors for type 2 diabetes include:  * **Prediabetes:** Having blood glucose levels that are higher than normal, but not high enough to be characterized as d
Best semantic similarity : 0.6853
Best evidence page       : 5
NLI entailment           : 0.0005
NLI neutral              : 0.9993
NLI contradiction        : 0.0002

Claim 2:
* **Age:** Risk increases with age [Source 2 | Page 9]
Best semantic similarity : 0.3677
Best evidence page       : 9
NLI entailment           : 0.0321
NLI neutral              : 0.9676
NLI contradiction        : 0.0003

Claim 3:
* **Overweight or obesity:** A body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source 2 | Page 9]
Best semantic similarity : 0.4749
Best evidence page       : 9
NLI entailment           : 0.0114
NLI neutral              : 0.9876
NLI contradiction        : 0.0009

Claim 4:
* **Family history of diabetes:** Having a parent or sibling with diabe

In [35]:
# ============================================
# STEP 6J-9 - SENTENCE-LEVEL EVIDENCE MATCHING
# ============================================

def split_into_sentences(text):
    """
    Split evidence text into individual sentences.
    """

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text.replace("\n", " ")
    )

    return [
        sentence.strip()
        for sentence in sentences
        if len(sentence.strip()) > 20
    ]


def calculate_consistency(
    answer,
    retrieved_documents,
    entailment_threshold=0.50
):
    """
    Calculate claim-level consistency using
    sentence-level evidence.

    Each answer claim is compared against individual
    evidence sentences from the retrieved documents.

    A claim is considered supported when at least one
    evidence sentence has an NLI entailment score
    above the threshold.
    """

    claims = split_into_claims(answer)

    if not claims or not retrieved_documents:
        return 0.0

    # Build evidence sentence pool
    evidence_sentences = []

    for doc in retrieved_documents:

        sentences = split_into_sentences(doc["text"])

        for sentence in sentences:
            evidence_sentences.append({
                "text": sentence,
                "page": doc["page"]
            })

    if not evidence_sentences:
        return 0.0

    supported_claims = 0

    for claim in claims:

        max_entailment = 0.0

        for evidence in evidence_sentences:

            entailment, neutral, contradiction = nli_score(
                evidence["text"],
                claim
            )

            max_entailment = max(
                max_entailment,
                entailment
            )

        if max_entailment >= entailment_threshold:
            supported_claims += 1

    consistency = (
        supported_claims / len(claims)
    )

    return float(consistency)


print("✅ Sentence-level consistency function created")

✅ Sentence-level consistency function created


In [36]:
# ============================================
# STEP 6J-10 - TEST SENTENCE-LEVEL CONSISTENCY
# ============================================

consistency = calculate_consistency(
    answer,
    retrieved_documents,
    entailment_threshold=0.50
)

print(
    f"🧩 Sentence-Level Consistency: "
    f"{consistency:.4f}"
)

🧩 Sentence-Level Consistency: 0.0000


In [37]:
# ============================================
# STEP 6J-11 - FINAL CLAIM-LEVEL CONSISTENCY
# ============================================

def clean_claim(claim):
    """
    Remove markdown formatting and source citations
    before performing evidence matching.
    """

    claim = re.sub(
        r'\[Source\s*\d+\s*\|\s*Page\s*\d+\]',
        '',
        claim,
        flags=re.IGNORECASE
    )

    claim = claim.replace("*", "")
    claim = claim.replace("•", "")

    claim = re.sub(r'\s+', ' ', claim)

    return claim.strip()


def calculate_consistency(
    answer,
    retrieved_documents,
    semantic_threshold=0.40,
    support_threshold=0.50
):
    """
    Calculate claim-level consistency.

    For every claim:
    1. Clean the claim.
    2. Find the most semantically relevant evidence sentence.
    3. Calculate NLI entailment against that evidence.
    4. Combine semantic relevance and NLI entailment.
    5. Mark the claim as supported when the combined
       claim-support score reaches the threshold.

    Consistency =
    supported claims / total claims

    Initial heuristic thresholds will be evaluated
    and tuned later using an evaluation dataset.
    """

    claims = split_into_claims(answer)

    if not claims or not retrieved_documents:
        return 0.0

    # Create evidence sentence pool
    evidence_sentences = []

    for doc in retrieved_documents:

        sentences = split_into_sentences(doc["text"])

        for sentence in sentences:

            evidence_sentences.append({
                "text": sentence,
                "page": doc["page"]
            })

    if not evidence_sentences:
        return 0.0

    supported_claims = 0

    for claim in claims:

        clean_claim_text = clean_claim(claim)

        claim_embedding = embedding_model.encode(
            clean_claim_text,
            normalize_embeddings=True
        )

        best_similarity = -1.0
        best_evidence = None

        # Find best semantic evidence
        for evidence in evidence_sentences:

            evidence_embedding = embedding_model.encode(
                evidence["text"],
                normalize_embeddings=True
            )

            similarity = float(
                np.dot(
                    claim_embedding,
                    evidence_embedding
                )
            )

            if similarity > best_similarity:

                best_similarity = similarity
                best_evidence = evidence

        if best_evidence is None:
            continue

        # NLI on the best matching evidence
        entailment, neutral, contradiction = nli_score(
            best_evidence["text"],
            clean_claim_text
        )

        # Combined claim support
        claim_support = (
            0.50 * best_similarity +
            0.50 * entailment
        )

        if (
            best_similarity >= semantic_threshold
            and claim_support >= support_threshold
        ):
            supported_claims += 1

    consistency = (
        supported_claims / len(claims)
    )

    return float(consistency)


print("✅ Final claim-level consistency function created")

✅ Final claim-level consistency function created


In [38]:
# ============================================
# STEP 6J-12 - TEST FINAL CONSISTENCY
# ============================================

consistency = calculate_consistency(
    answer,
    retrieved_documents
)

print(
    f"🧩 Final Consistency: "
    f"{consistency:.4f}"
)

🧩 Final Consistency: 0.3333


In [39]:
# ============================================
# STEP 6K - FINAL CONFIDENCE SCORE
# ============================================

final_confidence = calculate_final_confidence(
    semantic_similarity=notebook5_result["semantic_similarity"],
    nli_entailment=notebook5_result["nli_entailment"],
    retrieval_quality=retrieval_quality,
    consistency=consistency
)

confidence_score = final_confidence["confidence_score"]
confidence_level = final_confidence["confidence_level"]

print("📊 FINAL CONFIDENCE")
print("=" * 50)

print(
    f"Semantic Similarity : "
    f"{notebook5_result['semantic_similarity']:.4f}"
)

print(
    f"NLI Entailment      : "
    f"{notebook5_result['nli_entailment']:.4f}"
)

print(
    f"Retrieval Quality   : "
    f"{retrieval_quality:.4f}"
)

print(
    f"Consistency         : "
    f"{consistency:.4f}"
)

print("-" * 50)

print(
    f"Confidence Score    : "
    f"{confidence_score:.4f}"
)

print(
    f"Confidence Level    : "
    f"{confidence_level}"
)

print(
    f"Decision             : "
    f"{notebook5_result['decision']}"
)

📊 FINAL CONFIDENCE
Semantic Similarity : 0.8071
NLI Entailment      : 0.9905
Retrieval Quality   : 0.6488
Consistency         : 0.3333
--------------------------------------------------
Confidence Score    : 0.7357
Confidence Level    : MEDIUM
Decision             : SUPPORTED


In [40]:
# ============================================
# STEP 6L - FINAL STRUCTURED RESULT
# ============================================

source_pages = sorted(
    set(
        doc["page"]
        for doc in retrieved_documents
    )
)

final_result = {
    "question": question,
    "answer": answer,

    # Notebook 5 metrics
    "semantic_similarity": notebook5_result[
        "semantic_similarity"
    ],

    "nli_entailment": notebook5_result[
        "nli_entailment"
    ],

    "support_score": notebook5_result[
        "support_score"
    ],

    "decision": notebook5_result[
        "decision"
    ],

    # Notebook 6 metrics
    "retrieval_quality": retrieval_quality,
    "consistency": consistency,

    # Final confidence
    "confidence_score": confidence_score,
    "confidence_level": confidence_level,

    # Evidence sources
    "source_pages": source_pages
}

print("✅ Final structured result created")

✅ Final structured result created


In [41]:
# ============================================
# STEP 6M - DISPLAY FINAL RESULT
# ============================================

print("╔" + "═" * 58 + "╗")
print("║" + " FINAL MEDICAL RAG RESULT ".center(58) + "║")
print("╚" + "═" * 58 + "╝")

print(f"\nQuestion:")
print(final_result["question"])

print(f"\nAnswer:")
print(final_result["answer"])

print("\n" + "-" * 60)

print("Evidence & Verification:")
print(
    f"Semantic Similarity : "
    f"{final_result['semantic_similarity']:.4f}"
)

print(
    f"NLI Entailment      : "
    f"{final_result['nli_entailment']:.4f}"
)

print(
    f"Support Score       : "
    f"{final_result['support_score']:.4f}"
)

print(
    f"Retrieval Quality   : "
    f"{final_result['retrieval_quality']:.4f}"
)

print(
    f"Consistency         : "
    f"{final_result['consistency']:.4f}"
)

print("\n" + "-" * 60)

print(
    f"Confidence Score    : "
    f"{final_result['confidence_score']:.4f}"
)

print(
    f"Confidence Level    : "
    f"{final_result['confidence_level']}"
)

print(
    f"Hallucination Decision: "
    f"{final_result['decision']}"
)

print(
    f"Source Pages        : "
    f"{final_result['source_pages']}"
)

╔══════════════════════════════════════════════════════════╗
║                 FINAL MEDICAL RAG RESULT                 ║
╚══════════════════════════════════════════════════════════╝

Question:
What are the risk factors for diabetes?

Answer:
Based on the provided documents, the risk factors for type 2 diabetes include:

* **Prediabetes:** Having blood glucose levels that are higher than normal, but not high enough to be characterized as diabetes [Source 1 | Page 5].
* **Age:** Risk increases with age [Source 2 | Page 9].
* **Overweight or obesity:** A body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source 2 | Page 9].
* **Family history of diabetes:** Having a parent or sibling with diabetes [Source 2 | Page 9].
* **Race/ethnicity:** Being a member of a high-risk population, including African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, and Pacific Islander American [Source 2 | Page 9].
* **History of GDM** (gestational diabetes mell

In [42]:
# ============================================
# STEP 6N - TEST 1: SUPPORTED QUESTION
# ============================================

test_question = "What are the risk factors for diabetes?"

test_documents, test_similarities = retrieve_documents(
    test_question,
    top_k=4
)

test_answer = generate_rag_answer(
    test_question,
    test_documents
)

test_notebook5 = analyze_answer(
    test_answer,
    test_documents
)

test_retrieval_quality = calculate_retrieval_quality(
    test_similarities
)

test_consistency = calculate_consistency(
    test_answer,
    test_documents
)

test_confidence = calculate_final_confidence(
    semantic_similarity=test_notebook5["semantic_similarity"],
    nli_entailment=test_notebook5["nli_entailment"],
    retrieval_quality=test_retrieval_quality,
    consistency=test_consistency
)

print("🧪 TEST 1 — SUPPORTED QUESTION")
print("=" * 60)

print(f"Question: {test_question}")

print("\nAnswer:")
print(test_answer)

print("\nMetrics:")
print(
    f"Semantic Similarity : "
    f"{test_notebook5['semantic_similarity']:.4f}"
)

print(
    f"NLI Entailment      : "
    f"{test_notebook5['nli_entailment']:.4f}"
)

print(
    f"Retrieval Quality   : "
    f"{test_retrieval_quality:.4f}"
)

print(
    f"Consistency         : "
    f"{test_consistency:.4f}"
)

print(
    f"Confidence Score    : "
    f"{test_confidence['confidence_score']:.4f}"
)

print(
    f"Confidence Level    : "
    f"{test_confidence['confidence_level']}"
)

print(
    f"Decision             : "
    f"{test_notebook5['decision']}"
)

🧪 TEST 1 — SUPPORTED QUESTION
Question: What are the risk factors for diabetes?

Answer:
Based on the provided documents, the risk factors for type 2 diabetes include:

* **Age:** Risk increases with age [Source 2 | Page 9].
* **Weight:** Overweight or obesity, defined as a body mass index (BMI) ≥ 25 kg/m² (or ≥ 23 kg/m² for Asian Americans) [Source 2 | Page 9].
* **Family History:** Having a parent or sibling with diabetes [Source 2 | Page 9].
* **High-Risk Populations:** Being African American, Hispanic/Latino, American Indian, Alaska Native, Asian American, or Pacific Islander American [Source 2 | Page 9].
* **History of Gestational Diabetes:** History of GDM [Source 2 | Page 9].
* **Physical Inactivity** [Source 2 | Page 9].
* **Hypertension** [Source 2 | Page 9].
* **Prediabetes:** Having blood glucose levels higher than normal, but not high enough to be classified as diabetes [Source 1 | Page 5].
* **Emerging Risk Factors:** Obstructive sleep apnea and chronic sleep deprivation (

In [43]:
# ============================================
# STEP 6O - TEST 2: UNSUPPORTED QUESTION
# ============================================

test_question_2 = (
    "What is the treatment for diabetes according to this document?"
)

test_documents_2, test_similarities_2 = retrieve_documents(
    test_question_2,
    top_k=4
)

test_answer_2 = generate_rag_answer(
    test_question_2,
    test_documents_2
)

test_notebook5_2 = analyze_answer(
    test_answer_2,
    test_documents_2
)

test_retrieval_quality_2 = calculate_retrieval_quality(
    test_similarities_2
)

test_consistency_2 = calculate_consistency(
    test_answer_2,
    test_documents_2
)

test_confidence_2 = calculate_final_confidence(
    semantic_similarity=test_notebook5_2["semantic_similarity"],
    nli_entailment=test_notebook5_2["nli_entailment"],
    retrieval_quality=test_retrieval_quality_2,
    consistency=test_consistency_2
)

print("🧪 TEST 2 — UNSUPPORTED QUESTION")
print("=" * 60)

print(f"Question: {test_question_2}")

print("\nAnswer:")
print(test_answer_2)

print("\nMetrics:")

print(
    f"Semantic Similarity : "
    f"{test_notebook5_2['semantic_similarity']:.4f}"
)

print(
    f"NLI Entailment      : "
    f"{test_notebook5_2['nli_entailment']:.4f}"
)

print(
    f"Retrieval Quality   : "
    f"{test_retrieval_quality_2:.4f}"
)

print(
    f"Consistency         : "
    f"{test_consistency_2:.4f}"
)

print(
    f"Confidence Score    : "
    f"{test_confidence_2['confidence_score']:.4f}"
)

print(
    f"Confidence Level    : "
    f"{test_confidence_2['confidence_level']}"
)

print(
    f"Decision             : "
    f"{test_notebook5_2['decision']}"
)

🧪 TEST 2 — UNSUPPORTED QUESTION
Question: What is the treatment for diabetes according to this document?

Answer:
Based on the provided document, specific pharmacological treatments (such as medications) are not detailed, but the document outlines the following components and approaches for treating and managing diabetes:

* **Control of Blood Glucose:** Managing and controlling blood glucose is a central component of diabetes care, aimed at relieving immediate symptoms and reducing long-term complications (Page 47). For adults with poor glucose control (A1C > 9.0%), treatment to lower A1C to a mean of 7.5% improves quality of life and work productivity (Page 47).
* **Patient-Centered, Individualized Approach:** Care involves considering individual patient factors and preferences to establish individualized treatment goals and strategies that balance the potential benefits against the potential harms of glucose control (Page 4, Page 47).
* **Key Principles of Management:** Broader trea

In [44]:
# ============================================
# STEP 6O - TEST 2: TRUE UNSUPPORTED QUESTION
# ============================================

test_question_2 = "What is the capital of France?"

test_documents_2, test_similarities_2 = retrieve_documents(
    test_question_2,
    top_k=4
)

test_answer_2 = generate_rag_answer(
    test_question_2,
    test_documents_2
)

test_notebook5_2 = analyze_answer(
    test_answer_2,
    test_documents_2
)

test_retrieval_quality_2 = calculate_retrieval_quality(
    test_similarities_2
)

test_consistency_2 = calculate_consistency(
    test_answer_2,
    test_documents_2
)

test_confidence_2 = calculate_final_confidence(
    semantic_similarity=test_notebook5_2["semantic_similarity"],
    nli_entailment=test_notebook5_2["nli_entailment"],
    retrieval_quality=test_retrieval_quality_2,
    consistency=test_consistency_2
)

print("🧪 TEST 2 — TRUE UNSUPPORTED QUESTION")
print("=" * 60)

print(f"Question: {test_question_2}")

print("\nAnswer:")
print(test_answer_2)

print("\nMetrics:")

print(
    f"Semantic Similarity : "
    f"{test_notebook5_2['semantic_similarity']:.4f}"
)

print(
    f"NLI Entailment      : "
    f"{test_notebook5_2['nli_entailment']:.4f}"
)

print(
    f"Retrieval Quality   : "
    f"{test_retrieval_quality_2:.4f}"
)

print(
    f"Consistency         : "
    f"{test_consistency_2:.4f}"
)

print(
    f"Confidence Score    : "
    f"{test_confidence_2['confidence_score']:.4f}"
)

print(
    f"Confidence Level    : "
    f"{test_confidence_2['confidence_level']}"
)

print(
    f"Decision             : "
    f"{test_notebook5_2['decision']}"
)

🧪 TEST 2 — TRUE UNSUPPORTED QUESTION
Question: What is the capital of France?

Answer:
Based on the provided document evidence, there is not enough evidence to answer the question.

Metrics:
Semantic Similarity : 0.0750
NLI Entailment      : 0.0007
Retrieval Quality   : 0.0470
Consistency         : 0.0000
Confidence Score    : 0.0321
Confidence Level    : LOW
Decision             : CONTRADICTED


In [25]:
# ============================================
# STEP 6H-4 - COMPLETE HALLUCINATION ANALYSIS
# ============================================

def analyze_answer(answer, retrieved_documents):

    similarity_score, similarities = semantic_similarity(
        answer,
        retrieved_documents
    )

    nli_results = check_nli_against_documents(
        answer,
        retrieved_documents
    )

    entailment_score = get_max_entailment(
        nli_results
    )

    support_score = calculate_support_score(
        similarity_score,
        entailment_score
    )

    decision = detect_hallucination(
        support_score,
        nli_results
    )

    return {
        "semantic_similarity": similarity_score,
        "nli_entailment": entailment_score,
        "support_score": support_score,
        "decision": decision,
        "nli_results": nli_results
    }


print("✅ Complete Notebook 5 analysis pipeline ready")

✅ Complete Notebook 5 analysis pipeline ready


In [1]:
# ============================================
# STEP 1 - CONFIDENCE SCORE FUNCTION
# ============================================

def calculate_confidence(
    semantic_similarity,
    nli_entailment,
    retrieval_quality,
    consistency
):
    """
    Calculate the initial confidence score.

    Weights:
    - Semantic Similarity : 30%
    - NLI Entailment      : 30%
    - Retrieval Quality   : 20%
    - Consistency         : 20%

    These are initial heuristic weights.
    They will be evaluated and tuned later.
    """

    confidence = (
        0.30 * semantic_similarity +
        0.30 * nli_entailment +
        0.20 * retrieval_quality +
        0.20 * consistency
    )

    # Keep score between 0 and 1
    confidence = max(0.0, min(1.0, confidence))

    return confidence


print("✅ Confidence score function created")

✅ Confidence score function created


In [2]:
# ============================================
# STEP 2 - RETRIEVAL QUALITY
# ============================================

def calculate_retrieval_quality(similarities):
    """
    Calculate retrieval quality from FAISS similarity scores.

    The retrieved documents are expected to have similarity
    scores in the range [0, 1].

    We use the mean similarity of the retrieved documents
    as the initial retrieval quality heuristic.
    """

    if similarities is None or len(similarities) == 0:
        return 0.0

    similarities = np.asarray(similarities, dtype=float)

    retrieval_quality = float(np.mean(similarities))

    # Keep score between 0 and 1
    retrieval_quality = max(0.0, min(1.0, retrieval_quality))

    return retrieval_quality


print("✅ Retrieval quality function created")

✅ Retrieval quality function created


In [ ]:
# ============================================
# STEP 5 - CONFIDENCE LEVEL
# ============================================

def get_confidence_level(confidence):
    if confidence >= 0.80:
        return "HIGH"
    elif confidence >= 0.60:
        return "MEDIUM"
    else:
        return "LOW"


print("Confidence level function created successfully.")

Confidence level function created successfully.


In [ ]:
# ============================================
# STEP 6 - TEST CONFIDENCE SCORING
# ============================================

test_semantic_similarity = 0.85
test_nli_entailment = 0.90
test_support_score = 0.80

test_confidence = calculate_confidence(
    test_semantic_similarity,
    test_nli_entailment,
    test_support_score
)

test_level = get_confidence_level(test_confidence)

print("Semantic Similarity :", test_semantic_similarity)
print("NLI Entailment      :", test_nli_entailment)
print("Support Score       :", test_support_score)
print("Confidence Score    :", round(test_confidence, 3))
print("Confidence Level    :", test_level)

Semantic Similarity : 0.85
NLI Entailment      : 0.9
Support Score       : 0.8
Confidence Score    : 0.86
Confidence Level    : HIGH


In [ ]:
# ============================================
# STEP 7 - TEST DIFFERENT CONFIDENCE LEVELS
# ============================================

test_cases = [
    {
        "name": "High Confidence",
        "semantic": 0.90,
        "nli": 0.92,
        "support": 0.88
    },
    {
        "name": "Medium Confidence",
        "semantic": 0.70,
        "nli": 0.65,
        "support": 0.68
    },
    {
        "name": "Low Confidence",
        "semantic": 0.35,
        "nli": 0.40,
        "support": 0.30
    }
]

for case in test_cases:
    score = calculate_confidence(
        case["semantic"],
        case["nli"],
        case["support"]
    )

    level = get_confidence_level(score)

    print("\n", case["name"])
    print("Confidence Score :", round(score, 3))
    print("Confidence Level :", level)



 High Confidence
Confidence Score : 0.904
Confidence Level : HIGH

 Medium Confidence
Confidence Score : 0.676
Confidence Level : MEDIUM

 Low Confidence
Confidence Score : 0.36
Confidence Level : LOW


In [ ]:
# ============================================
# STEP 8 - CONNECT NOTEBOOK 5 WITH NOTEBOOK 6
# ============================================

def add_confidence_score(notebook5_result):
    """
    Add confidence score and confidence level
    to the result produced by Notebook 5.
    """

    confidence = calculate_confidence(
        notebook5_result["semantic_similarity"],
        notebook5_result["nli_entailment"],
        notebook5_result["support_score"]
    )

    level = get_confidence_level(confidence)

    result = notebook5_result.copy()

    result["confidence_score"] = confidence
    result["confidence_level"] = level

    return result


print("Notebook 5 → Notebook 6 connection created successfully.")

Notebook 5 → Notebook 6 connection created successfully.


In [ ]:
# ============================================
# STEP 9 - TEST NOTEBOOK 5 → NOTEBOOK 6
# ============================================

sample_notebook5_result = {
    "semantic_similarity": 0.85,
    "nli_entailment": 0.90,
    "support_score": 0.875,
    "decision": "SUPPORTED"
}

confidence_result = add_confidence_score(sample_notebook5_result)

print("========== CONFIDENCE RESULT ==========")
print("Semantic Similarity :", confidence_result["semantic_similarity"])
print("NLI Entailment      :", confidence_result["nli_entailment"])
print("Evidence Support    :", confidence_result["support_score"])
print("Detection Result    :", confidence_result["decision"])
print("Confidence Score    :", round(confidence_result["confidence_score"], 3))
print("Confidence Level    :", confidence_result["confidence_level"])

========== CONFIDENCE RESULT ==========
Semantic Similarity : 0.85
NLI Entailment      : 0.9
Evidence Support    : 0.875
Detection Result    : SUPPORTED
Confidence Score    : 0.875
Confidence Level    : HIGH


In [ ]:
# ============================================
# STEP 10 - FINAL CONFIDENCE REPORT
# ============================================

print("=" * 50)
print("        CONFIDENCE SCORING RESULT")
print("=" * 50)

print(f"Detection Result   : {confidence_result['decision']}")
print(f"Semantic Similarity: {confidence_result['semantic_similarity']:.4f}")
print(f"NLI Entailment     : {confidence_result['nli_entailment']:.4f}")
print(f"Evidence Support   : {confidence_result['support_score']:.4f}")
print(f"Confidence Score   : {confidence_result['confidence_score']:.4f}")
print(f"Confidence Level   : {confidence_result['confidence_level']}")

print("=" * 50)


        CONFIDENCE SCORING RESULT
Detection Result   : SUPPORTED
Semantic Similarity: 0.8500
NLI Entailment     : 0.9000
Evidence Support   : 0.8750
Confidence Score   : 0.8750
Confidence Level   : HIGH


In [ ]:
# ============================================
# STEP 11 - DECISION-AWARE CONFIDENCE
# ============================================

def finalize_confidence(notebook5_result):
    """
    Generate final confidence score and level
    while considering the hallucination detection decision.
    """

    confidence = calculate_confidence(
        notebook5_result["semantic_similarity"],
        notebook5_result["nli_entailment"],
        notebook5_result["support_score"]
    )

    decision = notebook5_result["decision"]

    # Contradicted answers should never receive HIGH confidence
    if decision == "CONTRADICTED":
        level = "LOW"
    else:
        level = get_confidence_level(confidence)

    result = notebook5_result.copy()
    result["confidence_score"] = confidence
    result["confidence_level"] = level

    return result


print("Decision-aware confidence function created successfully.")

Decision-aware confidence function created successfully.


In [ ]:
# ============================================
# STEP 12 - TEST DECISION-AWARE CONFIDENCE
# ============================================

test_results = [
    {
        "name": "Supported Answer",
        "semantic_similarity": 0.85,
        "nli_entailment": 0.90,
        "support_score": 0.875,
        "decision": "SUPPORTED"
    },
    {
        "name": "Potential Hallucination",
        "semantic_similarity": 0.55,
        "nli_entailment": 0.50,
        "support_score": 0.525,
        "decision": "POTENTIAL HALLUCINATION"
    },
    {
        "name": "Contradicted Answer",
        "semantic_similarity": 0.85,
        "nli_entailment": 0.90,
        "support_score": 0.875,
        "decision": "CONTRADICTED"
    }
]

for case in test_results:

    result = finalize_confidence(case)

    print("\n" + case["name"])
    print("Detection Result :", result["decision"])
    print("Confidence Score :", round(result["confidence_score"], 3))
    print("Confidence Level :", result["confidence_level"])


Supported Answer
Detection Result : SUPPORTED
Confidence Score : 0.875
Confidence Level : HIGH

Potential Hallucination
Detection Result : POTENTIAL HALLUCINATION
Confidence Score : 0.525
Confidence Level : LOW

Contradicted Answer
Detection Result : CONTRADICTED
Confidence Score : 0.875
Confidence Level : LOW


In [ ]:
# ============================================
# STEP 13 - FINAL CONFIDENCE OUTPUT
# ============================================

final_confidence_result = {
    "decision": confidence_result["decision"],
    "semantic_similarity": confidence_result["semantic_similarity"],
    "nli_entailment": confidence_result["nli_entailment"],
    "support_score": confidence_result["support_score"],
    "confidence_score": confidence_result["confidence_score"],
    "confidence_level": confidence_result["confidence_level"]
}

print("Final Notebook 6 output created successfully.")
print("\nFinal Confidence Result:")
print(final_confidence_result)

Final Notebook 6 output created successfully.

Final Confidence Result:
{'decision': 'SUPPORTED', 'semantic_similarity': 0.85, 'nli_entailment': 0.9, 'support_score': 0.875, 'confidence_score': 0.8750000000000001, 'confidence_level': 'HIGH'}


In [ ]:
# ============================================
# STEP 14 - FINAL CONFIDENCE SCORING PIPELINE
# ============================================

def confidence_scoring_pipeline(notebook5_result):
    """
    Convert Notebook 5 hallucination detection
    output into a final confidence result.
    """

    result = finalize_confidence(notebook5_result)

    return {
        "decision": result["decision"],
        "confidence_score": round(result["confidence_score"], 4),
        "confidence_level": result["confidence_level"]
    }


print("Final confidence scoring pipeline created successfully.")


Final confidence scoring pipeline created successfully.


In [ ]:
# ============================================
# STEP 15 - TEST FINAL CONFIDENCE PIPELINE
# ============================================

final_test = confidence_scoring_pipeline(sample_notebook5_result)

print("=" * 50)
print("       FINAL CONFIDENCE PIPELINE")
print("=" * 50)

print("Detection Result :", final_test["decision"])
print("Confidence Score :", final_test["confidence_score"])
print("Confidence Level :", final_test["confidence_level"])

print("=" * 50)

       FINAL CONFIDENCE PIPELINE
Detection Result : SUPPORTED
Confidence Score : 0.875
Confidence Level : HIGH


In [ ]:
# ============================================
# STEP 16 - FINAL RESULT DISPLAY
# ============================================

def display_confidence_result(result):
    print("=" * 50)
    print("       MEDICAL RAG CONFIDENCE RESULT")
    print("=" * 50)

    print(f"Detection Result   : {result['decision']}")
    print(f"Confidence Score   : {result['confidence_score']:.4f}")
    print(f"Confidence Level   : {result['confidence_level']}")

    print("=" * 50)


display_confidence_result(final_test)

       MEDICAL RAG CONFIDENCE RESULT
Detection Result   : SUPPORTED
Confidence Score   : 0.8750
Confidence Level   : HIGH


In [ ]:
# ============================================
# STEP 17 - STORE FINAL CONFIDENCE RESULT
# ============================================

notebook6_output = {
    "confidence_score": final_test["confidence_score"],
    "confidence_level": final_test["confidence_level"],
    "decision": final_test["decision"]
}

print("Notebook 6 output is ready for Notebook 7.")
print("\nNotebook 6 Output:")
print(notebook6_output)

Notebook 6 output is ready for Notebook 7.

Notebook 6 Output:
{'confidence_score': 0.875, 'confidence_level': 'HIGH', 'decision': 'SUPPORTED'}


In [ ]:
# ============================================
# STEP 18 - NOTEBOOK 6 VALIDATION
# ============================================

required_outputs = [
    "confidence_score",
    "confidence_level",
    "decision"
]

print("=" * 50)
print("       NOTEBOOK 6 VALIDATION")
print("=" * 50)

for output in required_outputs:
    if output in notebook6_output:
        print(f"✓ {output} available")
    else:
        print(f"✗ {output} missing")

print("\nNotebook 6 Confidence Scoring completed successfully.")
print("=" * 50)

       NOTEBOOK 6 VALIDATION
✓ confidence_score available
✓ confidence_level available
✓ decision available

Notebook 6 Confidence Scoring completed successfully.
